In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.decomposition import PCA

In [2]:
# ===================================================================
# PART 1: 기본 설정
# ===================================================================
# --- 입력 파일 및 디렉토리 ---
IMAGE_DIRECTORY = '../../data/interim/satellites/validation_satellites/rot_90' # ❗️ 이미지들이 있는 폴더
TABULAR_DATA_PATH = '../../data/interim/apt/validation_apt_with_long_lat.csv' # ❗️ 원본 매매 데이터

# --- 출력 파일 ---
OUTPUT_CSV_PATH = '../../data/final/validation/validation_img_features.csv' # ❗️ 최종 결과물이 저장될 경로

# --- 모델 파라미터 ---
PCA_COMPONENTS = 64
BATCH_SIZE = 64

In [3]:

# ===================================================================
# PART 2: 원본 데이터 및 이미지 경로 준비
# ===================================================================
print("PART 2: 원본 데이터 및 이미지 경로를 준비합니다...")
df = pd.read_csv(TABULAR_DATA_PATH)
df['계약일자'] = pd.to_datetime(df['계약일자'])
df = df.sort_values('계약일자').reset_index(drop=True)

# 가정: 이미지 파일 이름이 'apt_image_0.jpg', 'apt_image_1.jpg' ... 순서로 되어 있음
# 이 순서가 df의 행 순서와 일치한다고 가정합니다.
df['image_filename'] = [f'apt_image_{i}.jpg' for i in df.index]
df['image_path'] = df['image_filename'].apply(lambda filename: os.path.join(IMAGE_DIRECTORY, filename))

print(f"총 {len(df)}개의 데이터와 이미지 경로를 매칭했습니다.")
print("PART 2: 완료!")


PART 2: 원본 데이터 및 이미지 경로를 준비합니다...
총 98개의 데이터와 이미지 경로를 매칭했습니다.
PART 2: 완료!


In [5]:

# ===================================================================
# PART 3: 이미지 특징 추출 및 차원 축소 (ResNet-50 + PCA)
# ===================================================================
print("\nPART 3: 이미지 특징 추출 및 PCA 차원 축소를 시작합니다... (시간이 소요될 수 있습니다)")

def extract_and_reduce_image_features(image_paths, n_components, batch_size):
    # 1. ResNet-50 특징 추출기 모델 정의
    image_input = Input(shape=(224, 224, 3), name='image_input')
    base_cnn = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_tensor=image_input)
    base_cnn.trainable = False
    extractor_model = Model(inputs=image_input, outputs=base_cnn.output, name='resnet50_extractor')

    # 2. 이미지 데이터 제너레이터 정의
    def image_generator(paths, b_size):
        for i in range(0, len(paths), b_size):
            batch_paths = paths[i:i+b_size]
            batch_images = []
            for path in batch_paths:
                try:
                    img = load_img(path, target_size=(224, 224))
                    img_array = img_to_array(img)
                    batch_images.append(img_array)
                except (FileNotFoundError, IOError):
                    print(f"Warning: 파일을 찾을 수 없습니다 - {path}")
                    batch_images.append(np.zeros((224, 224, 3)))
            yield preprocess_input(np.array(batch_images))
    
    # 3. 모든 이미지에 대해 2048차원 특징 추출
    gen = image_generator(image_paths, batch_size)
    num_batches = int(np.ceil(len(image_paths) / batch_size))
    features_2048d = extractor_model.predict(gen, steps=num_batches, verbose=1)
    
    # 4. PCA를 사용하여 차원 축소
    pca = PCA(n_components=n_components)
    features_reduced = pca.fit_transform(features_2048d)
    
    print(f"PCA가 보존하는 원본 데이터의 분산량: {np.sum(pca.explained_variance_ratio_):.4f}")
    return features_reduced, pca

# 실제 특징 추출 실행
image_features_64d, pca_model = extract_and_reduce_image_features(
    image_paths=df['image_path'],
    n_components=PCA_COMPONENTS,
    batch_size=BATCH_SIZE
)
print("PART 3: 완료!")



PART 3: 이미지 특징 추출 및 PCA 차원 축소를 시작합니다... (시간이 소요될 수 있습니다)


2025-09-21 19:04:38.307683: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


2/2 [==============================] - 2s 705ms/step
PCA가 보존하는 원본 데이터의 분산량: 0.9920
PART 3: 완료!


In [ ]:

# ===================================================================
# PART 4: 원본 데이터와 이미지 특징 결합 및 저장
# ===================================================================
print("\nPART 4: 원본 데이터와 추출된 이미지 특징을 결합합니다...")

# 추출된 특징 벡터로 새로운 데이터프레임 생성
image_feature_cols = [f'img_pca_{i}' for i in range(PCA_COMPONENTS)]
df_image_features = pd.DataFrame(image_features_64d, columns=image_feature_cols, index=df.index)

# 원본 데이터프레임과 이미지 특징 데이터프레임 결합
df_combined = pd.concat([df, df_image_features], axis=1)

# 이미지 경로 등 불필요한 컬럼은 이제 제거 가능
df_combined = df_combined.drop(columns=['image_filename', 'image_path'])

# 최종 결과를 새로운 CSV 파일로 저장
df_combined.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')

print("PART 4: 완료!")
print(f"\n🎉 모든 과정이 완료되었습니다! 최종 데이터가 아래 경로에 저장되었습니다.")
print(OUTPUT_CSV_PATH)
print("\n저장된 데이터의 상위 5개 샘플:")
df_combined.head()



PART 4: 원본 데이터와 추출된 이미지 특징을 결합합니다...
PART 4: 완료!

🎉 모든 과정이 완료되었습니다! 최종 데이터가 아래 경로에 저장되었습니다.
../../data/final/validation/validation_img_features.csv

저장된 데이터의 상위 5개 샘플:
                단지명  전용면적(㎡)   층  건축년도        도로명  면적당 단가(만원)  아파트 나이  \
0             금호어울림    84.95   4  2003   도곡로7길 22    7.738692      22   
1       삼성동중앙하이츠빌리지    59.98  21  2004  학동로68길 30    8.349108      21   
2           강남데시앙포레    84.81   1  2014  광평로34길 55    7.737774      11   
3        성원대치2단지아파트    39.53   4  1992  개포로109길 9    8.157949      33   
4  푸른마을아파트101동~111동    59.76  10  1994  일원로14길 25    8.064443      31   

        계약일자          경도         위도  ...  img_pca_54  img_pca_55  img_pca_56  \
0 2025-07-01  127.034771  37.492374  ...   -0.525545   -0.066494   -0.493326   
1 2025-07-01  127.046510  37.516580  ...   -0.609705    0.181431   -0.095318   
2 2025-07-01  127.092172  37.480856  ...    0.554606   -0.278469   -0.538191   
3 2025-07-01  127.075101  37.495752  ...   -1.148804    0.239364   -0.89

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,계약일자,경도,위도,...,img_pca_54,img_pca_55,img_pca_56,img_pca_57,img_pca_58,img_pca_59,img_pca_60,img_pca_61,img_pca_62,img_pca_63
0,금호어울림,84.95,4,2003,도곡로7길 22,7.738692,22,2025-07-01,127.034771,37.492374,...,-0.525545,-0.066494,-0.493326,0.639484,0.207204,-0.797227,-0.755085,-0.382876,-0.253621,0.385204
1,삼성동중앙하이츠빌리지,59.98,21,2004,학동로68길 30,8.349108,21,2025-07-01,127.046510,37.516580,...,-0.609705,0.181431,-0.095318,-0.262441,-0.548418,-0.549298,-0.272017,0.418846,0.168552,-0.063344
2,강남데시앙포레,84.81,1,2014,광평로34길 55,7.737774,11,2025-07-01,127.092172,37.480856,...,0.554606,-0.278469,-0.538191,0.001475,-0.312570,0.200709,0.083886,0.313382,0.432236,-0.180675
3,성원대치2단지아파트,39.53,4,1992,개포로109길 9,8.157949,33,2025-07-01,127.075101,37.495752,...,-1.148804,0.239364,-0.895362,0.126959,-0.562291,1.165777,0.208531,-0.245579,0.349564,-0.807281
4,푸른마을아파트101동~111동,59.76,10,1994,일원로14길 25,8.064443,31,2025-07-01,127.080875,37.484761,...,0.331407,-0.209895,-0.158365,-0.668636,-0.251978,0.079327,-0.260716,0.908000,-0.247900,0.028212
